# ML Pipeline — Training, Batch Predict & Tests

> **Notebook flow:** 01 Setup → 02 EDA → **[03 Train]** → 04 Docker → 05 Kubernetes → 06 Cleanup · 07 Azure Deploy · 08 API Tests

This notebook runs the full ML pipeline locally and executes the test suite. Run cells top-to-bottom.

**Prerequisite:** Run `01_devcontainer_setup.ipynb` first to confirm the environment is ready.

---

## Pipeline stages
```
data → preprocessing → training → evaluation → artifact
       (src/data.py)  (src/features.py)  (src/train.py)  (src/evaluate.py)
                                                                  ↓
                                                     artifacts/model.pkl
                                                     artifacts/metrics.json
```


In [ ]:
import sys
import os

# Ensure project root is on the path
ROOT = "/workspaces/marketing-model-mlops-azure"
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(f"Working directory: {os.getcwd()}")

## 1. Train the Model

Runs the full pipeline: load → feature engineering → train → evaluate → save artifact.

**Outputs:**
- `artifacts/model.pkl` — trained sklearn Pipeline
- `artifacts/metrics.json` — ROC-AUC, F1, and additional metrics

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure
python main.py train

In [ ]:
# Verify artifacts were created
import json

model_path = os.path.join(ROOT, "artifacts/model.pkl")
metrics_path = os.path.join(ROOT, "artifacts/metrics.json")

print(f"model.pkl exists:   {os.path.exists(model_path)}")
print(f"metrics.json exists:{os.path.exists(metrics_path)}")

if os.path.exists(metrics_path):
    print()
    print("=== Metrics ===")
    with open(metrics_path) as f:
        metrics = json.load(f)
    for k, v in metrics.items():
        print(f"  {k}: {v}")

## 2. Inspect the Trained Model

Load and inspect the artifact to confirm it is a valid sklearn pipeline.

In [ ]:
import joblib

model = joblib.load(os.path.join(ROOT, "artifacts/model.pkl"))
print(f"Model type: {type(model)}")
print()
print("Pipeline steps:")
for name, step in model.steps:
    print(f"  {name}: {type(step).__name__}")

## 3. Batch Predict

Run inference on the full dataset CSV and save results to `data/results/predictions.csv`.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

# Print predictions to stdout (first 5 lines)
python main.py predict \
  --input data/raw/bank_marketing_data.csv \
  --output data/results/predictions.csv

echo ""
echo "Output file:"
head -5 data/results/predictions.csv

In [ ]:
# Inspect predictions in Python
import pandas as pd

preds = pd.read_csv(os.path.join(ROOT, "data/results/predictions.csv"))
print(f"Prediction rows: {len(preds)}")
print(f"Columns: {list(preds.columns)}")
print()
print(preds.head())

## 4. Run Training Tests

Runs all test modules covering the training pipeline. A trained `artifacts/model.pkl` is required — run **Section 1** first.

> API and prediction endpoint tests are covered in **`08_prediction_testing.ipynb`**.

| Test module | What it covers |
|---|---|
| `test_config.py` | Config loading, `get_config_path()`, required keys, missing file handling |
| `test_data.py` | DataFrame loading, row counts, column validation |
| `test_features.py` | Cleaning transforms, config-driven columns, split sizes, preprocessor construction, robustness (missing columns) |
| `test_evaluate.py` | Metrics dict, `get_model_path()`, `load_model()`, `save_metrics()` |
| `test_train.py` | Model registry, pipeline construction, `save_model()` |
| `test_storage.py` | Local backend, blob backend (mocked), all storage helpers |


In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure
python -m pytest \
  tests/test_config.py \
  tests/test_data.py \
  tests/test_features.py \
  tests/test_evaluate.py \
  tests/test_train.py \
  tests/test_storage.py \
  -v --tb=short


## 5. Run Individual Training Test Modules

Runs each training test module in sequence, showing each module's output separately with a pass/fail status.

**To run a single module:** comment out the others in the `MODULES` list in the cell below.
For example, to run only feature engineering tests, keep only `("test_features.py", ...)` uncommented.

> API and prediction endpoint tests are in **`08_prediction_testing.ipynb`** (not included here).


In [ ]:
import subprocess
import sys

ROOT = "/workspaces/marketing-model-mlops-azure"

# To run a single module: comment out the others in this list
MODULES = [
    ("test_config.py",   "Config Loading & Paths"),
    ("test_data.py",     "Data Loading & Validation"),
    ("test_features.py", "Feature Engineering & Robustness"),
    ("test_evaluate.py", "Metrics & Artifact Loading"),
    ("test_train.py",    "Model Registry & Training"),
    ("test_storage.py",  "Local & Blob Storage"),
]

results = {}
for module, label in MODULES:
    print(f"\n{'='*65}")
    print(f"  {label}  ({module})")
    print(f"{'='*65}")
    r = subprocess.run(
        [sys.executable, "-m", "pytest", f"tests/{module}", "-v", "--tb=short"],
        cwd=ROOT,
    )
    results[module] = "✅ PASSED" if r.returncode == 0 else "❌ FAILED"

print(f"\n{'='*65}")
print("  Results Summary")
print(f"{'='*65}")
for mod, status in results.items():
    print(f"  {status}  {mod}")


## 6. Direct Inference (Python)

Call the model directly without starting the API server — useful for quick debugging.

In [ ]:
import joblib
import pandas as _pd
import yaml as _yaml
from src.config import load_config
from src.features import clean_data

config = load_config(os.path.join(ROOT, "config.yaml"))
model = joblib.load(os.path.join(ROOT, "artifacts/model.pkl"))

# Auto-derive a sample inference row from the first row of the training CSV.
# This avoids hardcoding dataset-specific column names — works for any dataset.
with open(os.path.join(ROOT, "config.yaml")) as _f:
    _cfg = _yaml.safe_load(_f)

_raw_path = os.path.join(ROOT, _cfg["data"]["raw_path"])
_sep      = _cfg["data"].get("separator", ",")
_target   = _cfg["model"]["target_column"]

_row = _pd.read_csv(_raw_path, sep=_sep, nrows=1)
_feature_cols = [c for c in _row.columns if c != _target]
sample = _pd.DataFrame(
    [{k: (v.item() if hasattr(v, "item") else v) for k, v in _row[_feature_cols].iloc[0].items()}]
)

print(f"Sample row ({len(_feature_cols)} features): {_feature_cols}")

# Apply the same feature engineering the API applies before calling the Pipeline
# clean_data() derives engineered columns (e.g. contacted_before) and applies transforms
sample_clean = clean_data(sample, config)

pred  = model.predict(sample_clean)[0]
prob  = model.predict_proba(sample_clean)[0][1]
label = "yes" if pred == 1 else "no"

print(f"\nprediction:  {pred}")
print(f"probability: {prob:.4f}")
print(f"label:       {label}")


---

## Summary

| Step | Command | Expected result |
|---|---|---|
| Train | `python main.py train` | `artifacts/model.pkl` + `artifacts/metrics.json` created |
| Metrics | `artifacts/metrics.json` | ROC-AUC, F1 scores printed |
| Batch predict | `python main.py predict --input ... --output ...` | CSV with predictions |
| Training tests (all) | §4 cell — `pytest test_config test_data test_features test_evaluate test_train test_storage` | All training modules pass |
| Training tests (individual) | §5 cell — comment out all but one module in `MODULES` list | Target module passes |
| API / prediction tests | `08_prediction_testing.ipynb` | API endpoint tests pass |

Once training tests pass and the artifact is confirmed, open **`04_docker_testing.ipynb`** to build and test the Docker image.
